# <!-- TITLE --> [CETO2] - Détourage : isoler la nageoire avant de chercher le dauphin
<!-- DESC --> Le `Segmenter` de `dorsalmatch` retire la mer autour de la nageoire. On regarde ce qu'il fait et comment le régler.

## Décision de cette étape ML
Le ML reconnaît un dauphin à la **forme de sa nageoire** : encoches, cicatrices, bord de fuite. Sur une photo brute, la nageoire n'occupe souvent qu'une petite partie de l'image. Le reste, c'est de la mer, du ciel, parfois un bateau. Ce décor brouille l'empreinte calculée par le modèle.

Le détourage garde la nageoire, met le reste en blanc et recadre l'image sur elle.

## Ce que fait ce notebook
1. Il prend 6 photos du catalogue (3 côté droit, 3 côté gauche).
2. Il montre, pour chacune, la photo, le **masque** et le résultat.
3. Il fait varier deux réglages, la **marge** et le **seuil**, pour voir leur effet.

## Avant de lancer
- Les dépendances : `uv sync --group ml --group notebook`.
- Les photos : déjà en local si tu as fait `python -m pipeline.storage pull`. Sinon, le notebook télécharge seulement les 6 photos dont il a besoin. Il lui faut alors le fichier `.env` avec les accès au bucket.
- Au premier lancement, les poids du modèle (176 Mo) sont téléchargés depuis Hugging Face, puis gardés en cache.

## Step 1 - Initialisation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import dorsalmatch  # noqa: E402
from pipeline.storage import connect, load_env  # noqa: E402

# CONSTANTES
DATA = ROOT / "data"
CATALOGUE = "2026-11-06_catalogue_tt"
PHOTOS_PER_SIDE = 3

load_env(ROOT / ".env")

## Step 2 - Les photos

Le manifeste du catalogue liste une photo par côté et par dauphin. On écarte les photos signalées dans la colonne `anomalies`, puis on en tire 3 par côté. `random_state=0` donne toujours les mêmes : tout le monde regarde les mêmes images.

In [ ]:
bucket = None


# Le fichier en local, téléchargé depuis le bucket s'il manque
def local(path):
    global bucket
    if path.exists():
        return path
    if bucket is None:
        bucket = connect()
    return bucket.fetch(path, DATA)


manifest = pd.read_csv(
    local(DATA / "curated/catalogue/manifests" / f"{CATALOGUE}.csv"),
    dtype=str,
    keep_default_na=False,
)
usable = manifest[(manifest.anomalies == "") & manifest.cote.isin(["D", "G"])]
photos = usable.groupby("cote").sample(PHOTOS_PER_SIDE, random_state=0)
photos["chemin"] = [
    local(
        DATA
        / "curated/catalogue/photos"
        / CATALOGUE
        / row.individu
        / row.fichier_origine
    )
    for row in photos.itertuples()
]
photos[["individu", "cote", "largeur", "hauteur"]]

## Step 3 - Photo, masque, résultat

`Segmenter()` charge le modèle U-2-Net avec les réglages de [`dorsalmatch/segmenter.yaml`](../dorsalmatch/segmenter.yaml), dont l'adresse des poids sur Hugging Face.

Deux méthodes :
- `mask(photo)` : une image en noir et blanc de la taille de la photo. **Blanc = nageoire**, noir = fond.
- `remove_background(photo)` : la photo avec le fond en blanc, recadrée sur la nageoire.

Pour chaque photo, on affiche trois images :
1. le **masque** brut, en noir et blanc ;
2. le masque **en rouge sur la photo** : c'est la vue la plus pratique pour juger ce qui est gardé ;
3. la photo **détourée**.

**À regarder** : est-ce que le masque suit bien le contour de la nageoire ? Est-ce qu'il attrape autre chose (vague, bateau, deuxième dauphin) ? Sur ces 6 photos, tu devrais trouver au moins un deuxième dauphin gardé, des éclaboussures gardées et un dos coupé.

In [ ]:
segmenter = dorsalmatch.Segmenter()


def title(row):
    return f"{row.individu} - côté {row.cote}"


# Le masque en rouge sur la photo : la zone teintée est ce qui est gardé
def overlay(photo, mask):
    red = Image.new("RGB", photo.size, (255, 0, 0))
    return Image.composite(Image.blend(photo, red, 0.4), photo, mask)


fig, axes = plt.subplots(len(photos), 3, figsize=(15, 4 * len(photos)))
for axis_row, row in zip(axes, photos.itertuples(), strict=True):
    photo = Image.open(row.chemin).convert("RGB")
    mask = segmenter.mask(photo)
    images = [mask, overlay(photo, mask), segmenter.remove_background(photo)]
    names = ["masque", "masque sur la photo", "détourée"]
    for axis, image, name in zip(axis_row, images, names, strict=True):
        axis.imshow(image, cmap="gray")
        axis.set_title(f"{title(row)} : {name}")
        axis.axis("off")
plt.tight_layout()
plt.show()

## Step 4 - La marge

Le réseau calcule le masque sur une image réduite à 320 × 320 pixels, puis l'agrandit à la taille de la photo. Ce va-et-vient arrondit le contour : sans marge, le masque peut rogner **la pointe et le bord de fuite**, c'est-à-dire justement les zones qui identifient le dauphin.

`margin` élargit le masque tout autour de la nageoire, en fraction du plus grand côté de la photo. Mieux vaut garder un peu de mer que perdre des pixels de nageoire.

**À regarder** : la pointe de la nageoire avec `margin = 0`, puis avec la valeur par défaut (0,005).

In [ ]:
MARGINS = [0, 0.005, 0.02]
examples = photos.head(2)

fig, axes = plt.subplots(len(examples), len(MARGINS), figsize=(15, 5 * len(examples)))
for axis_row, row in zip(axes, examples.itertuples(), strict=True):
    for axis, margin in zip(axis_row, MARGINS, strict=True):
        segmenter = dorsalmatch.Segmenter({"margin": margin})
        axis.imshow(segmenter.remove_background(row.chemin))
        axis.set_title(f"{title(row)} : margin = {margin}")
        axis.axis("off")
plt.tight_layout()
plt.show()

## Step 5 - Le seuil

Pour chaque pixel, le réseau donne un score entre 0 et 1 : à quel point le pixel « ressort » de l'image. `threshold` coupe ce score en deux : au-dessus, c'est la nageoire, en dessous, c'est le fond.

- Seuil **bas** : le masque est généreux, il peut attraper des vagues.
- Seuil **haut** : le masque est strict, il peut perdre des bouts de nageoire.

**À regarder** : à partir de quel seuil le masque commence à perdre de la nageoire ?

In [ ]:
THRESHOLDS = [0.1, 0.3, 0.6]

fig, axes = plt.subplots(
    len(examples), len(THRESHOLDS), figsize=(15, 4 * len(examples))
)
for axis_row, row in zip(axes, examples.itertuples(), strict=True):
    for axis, threshold in zip(axis_row, THRESHOLDS, strict=True):
        segmenter = dorsalmatch.Segmenter({"threshold": threshold})
        photo = Image.open(row.chemin).convert("RGB")
        axis.imshow(overlay(photo, segmenter.mask(photo)))
        axis.set_title(f"{title(row)} : threshold = {threshold}")
        axis.axis("off")
plt.tight_layout()
plt.show()

## Votre défis ! 
Ce notebook montre que le détourage **fonctionne**. Il ne dit pas encore s'il **aide à retrouver le bon dauphin**. Pour le savoir, il faut relancer [la baseline](baseline_ml.ipynb) avec des empreintes calculées sur les photos détourées et comparer le top 10.

## Pour aller plus loin
Notebook de base --> [`remove-background-salient-object-detection`](remove-background-salient-object-detection.ipynb)